# Part 1: Monolingual Data Preprocessing + Visualization

Worked on by Naharin Siddiqui Nawar


**DATASET DESCRIPTION**


For the monolingual data, the lilithyu/kaggle-child-stories dataset was used, which contains short children’s stories written in English. The dataset consists of very brief narrative texts.

The dataset has 91,760 stories. After dropping 1,431 empty entries, it contains 212,910 sentences in total. It was counted by splitting each story into sentences with NLTK’s Punkt tokenizer.

The language in this corpus is simple and easy to read. Sentences are generally short and grammatically straightforward. The vocabulary is limited compared to technical writing.

Overall, the corpus is written in standard American English, with little regional variation. There is basically no slang, emojis, or social media/blog style writing. Also, the spelling/grammar stay pretty consistent. So, it is mostly clean and easy to tokenize. It is a good fit if we want monolingual data that is straightforward and fluent rather than technical or conversational.

Dataset: lilithyu/kaggle-child-stories (English children's stories)

This part:
1) Loads the dataset
2) Sentence-splits and tokenizes
3) Adds trigram boundary tokens
4) Replaces rare tokens


In [1]:
# @title Install + imports

!pip -q install datasets

import re
import json
from collections import Counter
from datasets import load_dataset
import matplotlib.pyplot as plt


In [2]:
# @title Dataset loading
from datasets import load_dataset
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt_tab")

ds = load_dataset("lilithyu/kaggle-child-stories")
data = ds["train"]
print(ds.keys())
print("Columns:", data.column_names)
print(data[0]["text"])
print(data[1]["text"])

total_sentences = 0
empty = 0
for t in data["text"]:
    if not t or not t.strip():
        empty += 1
        continue
    total_sentences += len(sent_tokenize(t))

print("Stories:", len(data))
print("Empty texts:", empty)
print("Total sentences:", total_sentences)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

cleaned_merged_fairy_tales_without_eos.t(…):   0%|          | 0.00/20.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/91760 [00:00<?, ? examples/s]

dict_keys(['train'])
Columns: ['text']
The Happy Prince.
HIGH above the city, on a tall column, stood the statue of the Happy Prince.  He was gilded all over with thin leaves of fine gold, for eyes he had two bright sapphires, and a large red ruby glowed on his sword-hilt.
Stories: 91760
Empty texts: 1431
Total sentences: 212910


## Regex rules
Sentence splitting:
- split after `. ! ?` followed by whitespace

Tokenization:
- words and numbers kept together
- punctuation is separated into its own token


In [3]:
import re
from collections import Counter
import matplotlib.pyplot as plt

LOWERCASE = True
MIN_COUNT_FOR_VOCAB = 2

sentence_split_regex = re.compile(r"(?<=[.!?])\s+")

token_regex = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?|[^\sA-Za-z0-9]")


In [4]:
# Note from Mimi: I edited this block of code to make preprocess a function that I could call later on
def preprocess(text, condition="none"):

  sentences_tokens = []

  for i in text:
      if condition == "training_data":
        text = i["text"]
      else:
        text = i

      if isinstance(text, str):

          text = text.strip()
          text = re.sub(r"\s+", " ", text)
          text = text.replace("’", "'")
          # lowercase
          if LOWERCASE:
              text = text.lower()

          # sentence split
          sentence_list = sentence_split_regex.split(text)

          sent_i = 0
          while sent_i < len(sentence_list):
              sent = sentence_list[sent_i].strip()

              if sent != "":
                  toks = token_regex.findall(sent)

                  if len(toks) > 0:
                      s = []
                      s.append("<s>")
                      s.append("<s>")

                      j = 0
                      while j < len(toks):
                          s.append(toks[j])
                          j += 1

                      s.append("</s>")
                      sentences_tokens.append(s)

              sent_i += 1

  return sentences_tokens

sentences_tokens = preprocess(data, "training_data")

print("Total sentences after preprocessing:", len(sentences_tokens))
if len(sentences_tokens) > 0:
   print("Example tokens:", sentences_tokens[0][:60])
else:
   print("There are no sentences.")

Total sentences after preprocessing: 211890
Example tokens: ['<s>', '<s>', 'the', 'happy', 'prince', '.', '</s>']


In [5]:
from collections import Counter

# Counting every token
token_counts = Counter()
for sent in sentences_tokens:
    for tok in sent:
        token_counts[tok] += 1


vocab = set()
for tok, c in token_counts.items():
    if c >= MIN_COUNT_FOR_VOCAB:
        vocab.add(tok)


vocab.add("<s>")
vocab.add("</s>")
vocab.add("<UNK>")

final_preprocessed_sentences = []
for sent in sentences_tokens:
    new_sent = []
    for tok in sent:
        if tok in vocab:
            new_sent.append(tok)
        else:
            new_sent.append("<UNK>")
    final_preprocessed_sentences.append(new_sent)

print("Vocab size:", len(vocab))
print("Example:", final_preprocessed_sentences[0][:60])


Vocab size: 32031
Example: ['<s>', '<s>', 'the', 'happy', 'prince', '.', '</s>']


In [6]:
def detokenize(tokens):

    cleaned = []
    i = 0
    while i < len(tokens):
        if tokens[i] != "<s>" and tokens[i] != "</s>":
            cleaned.append(tokens[i])
        i += 1

    out = []
    i = 0
    while i < len(cleaned):
        tok = cleaned[i]


        if tok in [".", ",", "!", "?", ";", ":", ")", "]", "}"]:
            if len(out) > 0:
                out[-1] = out[-1] + tok
            else:
                out.append(tok)
        else:
            out.append(tok)

        i += 1

    return " ".join(out)

k = 0
while k < 10 and k < len(final_preprocessed_sentences):
    print("\nExample", k + 1)
    print("TOKENS:", final_preprocessed_sentences[k][:60],
          ("..." if len(final_preprocessed_sentences[k]) > 60 else ""))
    print("TEXT:  ", detokenize(final_preprocessed_sentences[k]))
    k += 1




Example 1
TOKENS: ['<s>', '<s>', 'the', 'happy', 'prince', '.', '</s>'] 
TEXT:   the happy prince.

Example 2
TOKENS: ['<s>', '<s>', 'high', 'above', 'the', 'city', ',', 'on', 'a', 'tall', 'column', ',', 'stood', 'the', 'statue', 'of', 'the', 'happy', 'prince', '.', '</s>'] 
TEXT:   high above the city, on a tall column, stood the statue of the happy prince.

Example 3
TOKENS: ['<s>', '<s>', 'he', 'was', 'gilded', 'all', 'over', 'with', 'thin', 'leaves', 'of', 'fine', 'gold', ',', 'for', 'eyes', 'he', 'had', 'two', 'bright', 'sapphires', ',', 'and', 'a', 'large', 'red', 'ruby', 'glowed', 'on', 'his', 'sword', '-', 'hilt', '.', '</s>'] 
TEXT:   he was gilded all over with thin leaves of fine gold, for eyes he had two bright sapphires, and a large red ruby glowed on his sword - hilt.

Example 4
TOKENS: ['<s>', '<s>', 'he', 'was', 'very', 'much', 'admired', 'indeed', '.', '</s>'] 
TEXT:   he was very much admired indeed.

Example 5
TOKENS: ['<s>', '<s>', '“', 'he', 'is', 'as', 'beautiful

In [7]:
shown = 0
i = 0

while i < len(final_preprocessed_sentences) and shown < 10:
    after_sent = final_preprocessed_sentences[i]

    # checks if the sentence contains <UNK>
    has_unk = False
    j = 0
    while j < len(after_sent):
        if after_sent[j] == "<UNK>":
            has_unk = True
            break
        j += 1

    if has_unk:
        before_sent = sentences_tokens[i]
        print("\n==============================")
        print("Example with <UNK>:", shown + 1)

        print("\nBEFORE (<UNK> replacement):")
        print(before_sent)

        print("\nAFTER (<UNK> replacement):")
        print(after_sent)

        shown += 1

    i += 1

if shown == 0:
    print("No sentences with <UNK> found.")



Example with <UNK>: 1

BEFORE (<UNK> replacement):
['<s>', '<s>', '“', 'when', 'i', 'was', 'alive', 'and', 'had', 'a', 'human', 'heart', ',', '”', 'answered', 'the', 'statue', ',', '“', 'i', 'did', 'not', 'know', 'what', 'tears', 'were', ',', 'for', 'i', 'lived', 'in', 'the', 'palace', 'of', 'sans', '-', 'souci', ',', 'where', 'sorrow', 'is', 'not', 'allowed', 'to', 'enter', '.', '</s>']

AFTER (<UNK> replacement):
['<s>', '<s>', '“', 'when', 'i', 'was', 'alive', 'and', 'had', 'a', 'human', 'heart', ',', '”', 'answered', 'the', 'statue', ',', '“', 'i', 'did', 'not', 'know', 'what', 'tears', 'were', ',', 'for', 'i', 'lived', 'in', 'the', 'palace', 'of', '<UNK>', '-', '<UNK>', ',', 'where', 'sorrow', 'is', 'not', 'allowed', 'to', 'enter', '.', '</s>']

Example with <UNK>: 2

BEFORE (<UNK> replacement):
['<s>', '<s>', 'he', 'passed', 'over', 'the', 'ghetto', ',', 'and', 'saw', 'the', 'old', 'jews', 'bargaining', 'with', 'each', 'other', ',', 'and', 'weighing', 'out', 'money', 'in', 'copp

#Part 2 : Build Trigram Model with Stupid Backoff Smoothing and Storing the Model

### Worked on by Namratha Kashipurada Narendra

I have used the approach of storing counts over conditional probabilities. I am storing unigram, bigram and trigram counts and I compute conditional probability on the fly. I have a method called backoff_probabilities where I have computed conditional probabability. In my model, I store counts and use those to compute perplexity and log probability. I don't see it being efficient because storing so many probabilities takes huge memory. While implementing, I first did it the conditional probability way it took almost 12 minutes to just store the model.

### Smoothing Technique : Stupid Backoff with alpha = 0.4
I have used this technique because it's one of the easiest ways of Smoothing for an n gram model. Also we have used <'UNK'> for words that are rare and replaced it in the pre processing step.   

In [8]:
import math
import random
import pickle
from collections import Counter

def build_trigram_model(final_preprocessed_sentences, vocab):
  """
  This method will find the counts of unigrams, bigrams, trigrams and saves it.
  It takes the above preprocessed sentences as input. I have written the format below so that I can code quickly.
  final_preprocessed_sentences = list[ ['<s>','<s>','word','word','</s>]
  """
  unigram = Counter()
  bigram = Counter()
  trigram = Counter()
  total_tokens = 0

  for sentence in final_preprocessed_sentences:
    # Counting the number of unigrams
    start_index =0
    while start_index < len(sentence):
      unigram[sentence[start_index]] += 1
      total_tokens += 1
      start_index += 1

    # Counting the number of bigrams
    start_index = 1
    while start_index < len(sentence):
      bigram[(sentence[start_index-1],sentence[start_index])] += 1
      start_index += 1

    # Counting the numer of Trigrams
    start_index = 2
    while start_index < len( sentence):
      trigram[(sentence[start_index-2],sentence[start_index-1],sentence[start_index])] += 1
      start_index += 1


  model = {
      "vocab" : set(vocab),
      "unigram" : unigram,
      "bigram" : bigram,
      "trigram" : trigram,
      "total_tokens" : total_tokens
  }
  return model



In [9]:
def normalize_token(model, tok):
    return tok if tok in model["vocab"] else "<UNK>"

alpha = 0.4 # sb factor
def backoff_probabilities(model, word3, word2, word1):
  word3 = normalize_token(model,word3)
  word2 = normalize_token(model, word2)
  word1 = normalize_token(model, word1)
  unigram = model["unigram"]
  bigram = model["bigram"]
  trigram = model["trigram"]
  total_tokens = model["total_tokens"]

  search_tri_key = (word1, word2, word3)
  tri_phrase = trigram.get(search_tri_key,0)
  if tri_phrase > 0:
    bi_phrase = bigram.get((word1, word2),0)
    if bi_phrase > 0:
      return tri_phrase / bi_phrase

  search_bi_key = (word2, word3)
  bi_phrase = bigram.get(search_bi_key,0)
  if bi_phrase > 0:
    uni_phrase = unigram.get(word2,0)
    if uni_phrase > 0:
      return alpha * (bi_phrase/uni_phrase)

  uni_phrase = unigram.get(word3,0)
  if uni_phrase > 0 and total_tokens > 0:
    return (alpha ** 2) * (uni_phrase/ total_tokens)

# return epsilon, a small value othwrwise
  return 1e-12

In [10]:
def score_trigram(model , tokens):
  if not isinstance(model["vocab"], set):
        model["vocab"] = model["vocab"]

  # So the below line is to make sure we don't get weird values for too short sentences.
  if len(tokens) <3:
    return float("-inf"), float("inf")

  log_prob = 0.0
  M = 0 #number of predicted tokens

  for i in range(2, len(tokens)):
    word1 = tokens[i-2]
    word2 = tokens[i-1]
    word3 = tokens[i]
    p = backoff_probabilities( model, word3, word2, word1)
    log_prob += math.log(p)
    M += 1
    i += 1

  ppl = math.exp(-log_prob / max(1,M))
  return log_prob, ppl


In [11]:
def save_trigram_model(model, path):
    with open(path, "wb") as f:
        pickle.dump(model, f)

def load_trigram_model(path):
    with open(path, "rb") as f:
        return pickle.load(f)

In [12]:
# Training
model = build_trigram_model(final_preprocessed_sentences, vocab)

# Save the model
save_trigram_model(model, "trigram_model.pkl")

# Load the model later
model2 = load_trigram_model("trigram_model.pkl")

# Score an actual sentence perplexity
logp, ppl = score_trigram(model2, final_preprocessed_sentences[0])
print("Real sentence perplexity:", ppl)
print("Conditional Log Prob :",logp)


Real sentence perplexity: 12.378666264655424
Conditional Log Prob : -12.579872641939183


#Part 3: Evaluation

Worked on by Mimi Rapoport

##Evaluation 1: Evaluating the likelihood of sentences

Collecting five sentences to evaluate

In [13]:
# There is a running joke that native Yiddish speakers will say this sentence.
sentence1 = "Throw me out the window a towel."
# This is a nonsensical sentence that someone texted me when they were drunk:
sentence2 = "Let me but you asenin"
# This is a sentence from an academic paper I am currently writing:
senetence3 = "In a 1994 issue of Social Epistemology, Judith Genova published a paper challenging canonical readings of the Turing test, which Turing proposed in his 1950 paper “Computing Machinery and Intelligence” in place of the less answerable question of whether machines can think."
# This is a sentence that I wrote with the intention of making no sense:
sentence4 = "Who loving is, to date fig refridgerator and, no sheet but!"
# This is a meta use of the sentence that I am now writing:
sentence5 = "This is a meta use of the sentence that I am now writing:"
# This is a sentence that I wrote to fit the theme of the training data:
sentence6 = "The prince and the princess lived happily ever after."
# This is a sentence copied from the training data
sentence7 = "While the prince was staying at the palace he saw his sister, who greeted him with smiles and kisses."
# This is a common sentence
sentence8 = "How are you?"

sentences = [sentence1, sentence2, senetence3, sentence4, sentence5, sentence6, sentence7, sentence8]



Tokenizing the sentences

In [14]:
sentences_tokens_e = (preprocess(sentences, "none"))

Evaluate sentences:

In [15]:
evaluation = []
for i in sentences_tokens_e:
  logp, ppl = score_trigram(model2, i)
  evaluation.append((i, logp, ppl))

Order setntences by perplexity

In [16]:
sorted_by_ppl = sorted(evaluation, key=lambda choose: choose[2])
for i in sorted_by_ppl:
  print(i[0])
  print("ppl: ", i[2])

['<s>', '<s>', 'the', 'prince', 'and', 'the', 'princess', 'lived', 'happily', 'ever', 'after', '.', '</s>']
ppl:  13.357879881778873
['<s>', '<s>', 'how', 'are', 'you', '?', '</s>']
ppl:  14.675126754981745
['<s>', '<s>', 'while', 'the', 'prince', 'was', 'staying', 'at', 'the', 'palace', 'he', 'saw', 'his', 'sister', ',', 'who', 'greeted', 'him', 'with', 'smiles', 'and', 'kisses', '.', '</s>']
ppl:  19.8528027889936
['<s>', '<s>', 'this', 'is', 'a', 'meta', 'use', 'of', 'the', 'sentence', 'that', 'i', 'am', 'now', 'writing', ':', '</s>']
ppl:  126.89188450907663
['<s>', '<s>', 'throw', 'me', 'out', 'the', 'window', 'a', 'towel', '.', '</s>']
ppl:  134.67866558062101
['<s>', '<s>', 'let', 'me', 'but', 'you', 'asenin', '</s>']
ppl:  173.62564255420617
['<s>', '<s>', 'in', 'a', '1994', 'issue', 'of', 'social', 'epistemology', ',', 'judith', 'genova', 'published', 'a', 'paper', 'challenging', 'canonical', 'readings', 'of', 'the', 'turing', 'test', ',', 'which', 'turing', 'proposed', 'in', 

It is not surprising that the sentence copied from the training data and the sentence written to match the theme of the training data were given a low perplexity score.

It is similarly not surprising that the academic sentence was given a high perplexity score given that many of the words in that sentence likely didn't appear in the training data.

I was glad to see that the sentence I wrote to try to make the least sense possible was indeed given the highest perplexity score and that the very common sentence was given a low perplexity score.

##Evaluation 2: Sampling Sentences


Create conditional probability table

In [17]:
conditional_probability_dict = {}

# For every trigram
for key in list(model2["trigram"].keys()):
  # Get the beginnning bigram within the trigram
  bigram = (key[0], key[1])
  # Get the times the bigram appears
  bi_count = model2["bigram"][bigram]
  # Initialize the dictionary for the bigram if it doesn't exist
  if bigram not in conditional_probability_dict:
      conditional_probability_dict[bigram] = {}
  # Get third token in trigram
  third = key[2]
  # Get number of times that trigram appears
  third_count = model2["trigram"][key]
  # Get the conditional probability for that trigram
  probs = third_count / bi_count
  conditional_probability_dict[bigram][third] = probs

Create function to sort possible next token by relative frequency

In [18]:
def sort_options(bigram):
  sorted_probs = sorted(conditional_probability_dict[bigram].items(), key=lambda item: item[1], reverse = True)
  return sorted_probs

Creat function to get next token

In [19]:
import random

def get_next_token(bigram, type):
  if type == "textbook":
    random_float = random.random()
  elif type == "more likely":
    random_float = random.uniform(0.0, 0.5)
  iterator = 0
  addition = 0
  sorted_probs = sort_options(bigram)
  while addition < random_float:
    addition = addition + sorted_probs[iterator][1]
    next_token = sorted_probs[iterator-1][0]
    iterator = iterator + 1
  return next_token

Sample sentences

In [20]:
def sample_sentence(number, type):

  for i in range(number):

    bigram = ('<s>', '<s>')
    next_token = ''
    sentence = bigram

    # While haven't reached the end of sentence
    while next_token != '</s>':
      # Get next token
      next_token = get_next_token(bigram, type)
      bigram = (bigram[1], next_token)
      sentence = sentence + (next_token,)

    print(sentence)
    print(detokenize(sentence))
    print("\n")

Sample sentences using method from Speech and Language Processing textbook


In [21]:
sample_sentence(10, "textbook")

('<s>', '<s>', 'i', 'was', 'so', 'weak', 'in', 'the', 'gully', ',', 'falling', 'on', 'his', 'voyage', 'home', 'without', 'going', 'over', 'and', 'over', 'a', 'small', 'army', '.', '"', 'joe', 'had', 'sold', 'his', 'busts', ',', 'and', 'he', 'wore', 'this', 'suit', 'you', ',', 'my', 'mamma', 'went', 'on', ':', '“', 'you', "don't", 'mind', 'doing', 'a', 'heartless', 'and', 'obdurate', ';', 'and', 'elinor', 'tried', 'to', 'raise', 'a', 'dance', 'the', 'elephant', 'dance', '?', '”', 'if', 'the', 'blue', 'seas', 'that', 'are', 'never', 'troubled', 'me', 'very', 'well', 'give', 'her', 'what', 'she', 'meant', ';', 'for', ',', 'then', 'sam', 'wore', "'", 'em', 'they', 'are', 'stout', ',', 'warm', ',', 'very', 'quick', 'scent', 'for', 'anything', 'diamonds', 'or', 'bags', 'of', 'gems', 'in', 'the', 'street', 'and', 'not', 'to', 'twinkle', 'very', 'little', 'piece', ',', 'which', 'lifted', 'his', 'faded', 'hat', 'with', 'white', ',', 'the', 'poll', 'of', 'the', 'name', 'again', ',', 'lady', 'mar

Sample sentences using method from Speech and Language Processing textbook but constraining random float range to be between 0 and 0.5

In [22]:
sample_sentence(10, "more likely")

('<s>', '<s>', '“', 'the', "king's", 'son', 'had', 'been', 'done', ',', 'she', 'found', 'that', 'it', 'cracked', 'without', 'intending', 'to', 'go', ',', 'fust', 'and', 'foremost', '.', '</s>')
“ the king's son had been done, she found that it cracked without intending to go, fust and foremost.


('<s>', '<s>', 'twilight', 'was', 'upon', 'earth', 'again', 'he', 'said', ',', 'gave', 'mr', '.', '</s>')
twilight was upon earth again he said, gave mr.


('<s>', '<s>', 'twilight', 'was', 'upon', 'him', 'could', 'not', 'help', 'asking', 'again', '.', ']', '"', 'but', 'folks', 'has', 'a', 'great', 'hunger', 'for', 'gold', ',', 'but', 'the', 'little', 'maiden', ',', 'waited', 'patiently', 'on', 'and', 'retainers', 'were', 'the', 'four', 'little', 'trout', 'and', 'the', 'key', ',', 'what', '?', '.', '.', '!', '</s>')
twilight was upon him could not help asking again.] " but folks has a great hunger for gold, but the little maiden, waited patiently on and retainers were the four little trout and